<a href="https://colab.research.google.com/github/Vigneshvar16/NLP_DSA/blob/main/CO2_AT1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Q1: Morphological analysis pipeline for a document indexing system
# Words: connected, connecting, connection   (runs as-is in Google Colab)

words = ["connected", "connecting", "connection"]

# (suffix, type, meaning) - checked in this order
SUFFIX_RULES = [
    ("ing", "Inflectional", "present participle (-ING)"),
    ("ed",  "Inflectional", "past tense / past participle (-ED)"),
    ("ion", "Derivational", "verb -> noun (-ION)"),
]

def analyse(word):
    """Step 1 and 2: decompose the word into root + suffix and classify the suffix."""
    for suffix, kind, meaning in SUFFIX_RULES:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            root = word[:-len(suffix)]
            return root, suffix, kind, meaning
    return word, "-", "None", "already a base form"

def normalize(root):
    """Step 3: map every variant to one common base form."""
    return root.lower()

def print_table(headers, rows):
    widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    fmt = "| " + " | ".join("{:<%d}" % w for w in widths) + " |"
    print(line); print(fmt.format(*headers)); print(line)
    for r in rows:
        print(fmt.format(*[str(x) for x in r]))
    print(line)

rows = []
index = {}   # inverted-index style grouping: base form -> variants
for w in words:
    root, suffix, kind, meaning = analyse(w)
    base = normalize(root)
    parsed = f"{root} + -{suffix}" if suffix != "-" else root
    rows.append([w, root, "-" + suffix, parsed, kind, meaning, base])
    index.setdefault(base, []).append(w)

print("MORPHOLOGICAL ANALYSIS PIPELINE - DOCUMENT INDEXING\n")
print_table(["Word", "Root", "Suffix", "Parsed Structure", "Type", "Suffix Meaning", "Normalized"], rows)

print("\nIndex entries (one term retrieves all variants):")
for base, variants in index.items():
    print(f"  '{base}'  ->  {variants}")

MORPHOLOGICAL ANALYSIS PIPELINE - DOCUMENT INDEXING

+------------+---------+--------+------------------+--------------+------------------------------------+------------+
| Word       | Root    | Suffix | Parsed Structure | Type         | Suffix Meaning                     | Normalized |
+------------+---------+--------+------------------+--------------+------------------------------------+------------+
| connected  | connect | -ed    | connect + -ed    | Inflectional | past tense / past participle (-ED) | connect    |
| connecting | connect | -ing   | connect + -ing   | Inflectional | present participle (-ING)          | connect    |
| connection | connect | -ion   | connect + -ion   | Derivational | verb -> noun (-ION)                | connect    |
+------------+---------+--------+------------------+--------------+------------------------------------+------------+

Index entries (one term retrieves all variants):
  'connect'  ->  ['connected', 'connecting', 'connection']


In [3]:
# Q2: Morphological parsing module for sentiment analysis
# Words: unhappy, happiness, happily   (runs as-is in Google Colab)

words = ["unhappy", "happiness", "happily"]
LEXICON = {"happy"}        # known base words (used to validate each split)

PREFIX_RULES = {"un": ("Derivational", "negation: flips sentiment polarity")}
# (surface suffix, base-form ending restored, meaning)
SUFFIX_RULES = [
    ("iness", "y", "adjective -> noun (state/quality)"),
    ("ily",   "y", "adjective -> adverb (manner)"),
    ("ness",  "",  "adjective -> noun (state/quality)"),
    ("ly",    "",  "adjective -> adverb (manner)"),
]

def parse(word):
    prefix, suffix, base, changes = "", "", word, []
    # 1) strip prefix if the remainder is a valid base word
    for p in PREFIX_RULES:
        if word.startswith(p) and (word[len(p):] in LEXICON):
            prefix, base = p, word[len(p):]
    # 2) strip suffix, restoring spelling (happi -> happy) before checking the lexicon
    for suf, restore, meaning in SUFFIX_RULES:
        if base.endswith(suf):
            candidate = base[:-len(suf)] + restore
            if candidate in LEXICON:
                suffix, base = suf, candidate
                if restore:
                    changes.append("spelling change i -> y")
                break
    return prefix, base, suffix, changes

def sentiment(prefix):
    return "Negative (negated)" if prefix == "un" else "Positive (preserved)"

def print_table(headers, rows):
    widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    fmt = "| " + " | ".join("{:<%d}" % w for w in widths) + " |"
    print(line); print(fmt.format(*headers)); print(line)
    for r in rows:
        print(fmt.format(*[str(x) for x in r]))
    print(line)

rows = []
for w in words:
    prefix, base, suffix, changes = parse(w)
    parts = []
    if prefix: parts.append(prefix + "- (prefix)")
    parts.append(base + " (base)")
    if suffix: parts.append("-" + suffix + " (suffix)")
    kinds = []
    if prefix: kinds.append("prefix " + PREFIX_RULES[prefix][0])
    if suffix: kinds.append("suffix Derivational")
    rows.append([w, " + ".join(parts), kinds and " / ".join(kinds) or "None",
                 ", ".join(changes) or "-", sentiment(prefix), base])

print("MORPHOLOGICAL PARSING MODULE - SENTIMENT ANALYSIS\n")
print_table(["Word", "Morphological Breakdown", "Classification", "Spelling Rule",
             "Sentiment Effect", "Normalized Root"], rows)
print("\nAll three words share the root 'happy'. Only the prefix 'un-' reverses polarity;")
print("-iness (-ness) and -ily (-ly) change the part of speech but keep the positive meaning.")

MORPHOLOGICAL PARSING MODULE - SENTIMENT ANALYSIS

+-----------+--------------------------------+---------------------+------------------------+----------------------+-----------------+
| Word      | Morphological Breakdown        | Classification      | Spelling Rule          | Sentiment Effect     | Normalized Root |
+-----------+--------------------------------+---------------------+------------------------+----------------------+-----------------+
| unhappy   | un- (prefix) + happy (base)    | prefix Derivational | -                      | Negative (negated)   | happy           |
| happiness | happy (base) + -iness (suffix) | suffix Derivational | spelling change i -> y | Positive (preserved) | happy           |
| happily   | happy (base) + -ily (suffix)   | suffix Derivational | spelling change i -> y | Positive (preserved) | happy           |
+-----------+--------------------------------+---------------------+------------------------+----------------------+-----------------+

All

In [4]:
# Q3: Stemming-based preprocessing module for a search engine
# Words: played, player, playing   (runs as-is in Google Colab)

words = ["played", "player", "playing"]

# (suffix, type, what it does)
RULES = [
    ("ing", "Inflectional (grammatical)",   "verb -> present participle, POS unchanged"),
    ("ed",  "Inflectional (grammatical)",   "verb -> past tense, POS unchanged"),
    ("er",  "Derivational (word formation)", "verb -> noun (agent: one who plays), POS changes"),
]

def stem(word):
    for suffix, kind, note in RULES:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[:-len(suffix)], suffix, kind, note
    return word, "-", "None", "no affix found"

def print_table(headers, rows):
    widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    fmt = "| " + " | ".join("{:<%d}" % w for w in widths) + " |"
    print(line); print(fmt.format(*headers)); print(line)
    for r in rows:
        print(fmt.format(*[str(x) for x in r]))
    print(line)

rows, stems = [], set()
for w in words:
    s, affix, kind, note = stem(w)
    stems.add(s)
    rows.append([w, s, "-" + affix if affix != "-" else "-", kind, note, s])

print("STEMMING-BASED PREPROCESSING MODULE - SEARCH ENGINE\n")
print_table(["Original", "Stem", "Removed Affix", "Transformation Type", "Effect", "Index Term"], rows)

print("\nConsistent root extraction:", "YES" if len(stems) == 1 else "NO", "->", stems)
print("A query for 'playing' now also matches documents containing 'played' and 'player'.")

STEMMING-BASED PREPROCESSING MODULE - SEARCH ENGINE

+----------+------+---------------+-------------------------------+--------------------------------------------------+------------+
| Original | Stem | Removed Affix | Transformation Type           | Effect                                           | Index Term |
+----------+------+---------------+-------------------------------+--------------------------------------------------+------------+
| played   | play | -ed           | Inflectional (grammatical)    | verb -> past tense, POS unchanged                | play       |
| player   | play | -er           | Derivational (word formation) | verb -> noun (agent: one who plays), POS changes | play       |
| playing  | play | -ing          | Inflectional (grammatical)    | verb -> present participle, POS unchanged        | play       |
+----------+------+---------------+-------------------------------+--------------------------------------------------+------------+

Consistent root extrac

In [5]:
# Q4: Finite-state morphological parser for verb inflection
# Words: writes, writing, written   (runs as-is in Google Colab)

words = ["writes", "writing", "written"]

# Transition table: (state, input letter) -> next state.
# "writ" is the shared stem; 'e' gives the base form 'write'.
TRANSITIONS = {
    ("q0", "w"): "q1", ("q1", "r"): "q2", ("q2", "i"): "q3", ("q3", "t"): "q4",   # stem 'writ'
    ("q4", "e"): "q5",                                   # write (base, e restored)
    ("q5", "s"): "q6",                                   # write + s   (regular 3rd person singular)
    ("q4", "i"): "q7", ("q7", "n"): "q8", ("q8", "g"): "q9",   # writ + ing (regular, e-deletion)
    ("q4", "t"): "q10", ("q10", "e"): "q11", ("q11", "n"): "q12",  # writ + t + en (irregular participle)
}

# Accepting states -> (inflection tag, regular/irregular, morphemes, description)
ACCEPT = {
    "q6":  ("VBZ", "Regular",   "write + -s",   "3rd person singular present"),
    "q9":  ("VBG", "Regular",   "write + -ing", "present participle (final 'e' dropped before -ing)"),
    "q12": ("VBN", "Irregular", "write + -en",  "past participle (stem change: writ + t + en)"),
}
ROOT = "write"

def parse(word):
    state, path = "q0", ["q0"]
    for ch in word:
        nxt = TRANSITIONS.get((state, ch))
        if nxt is None:
            return None, path + ["REJECT"]
        state = nxt
        path.append(state)
    return (state if state in ACCEPT else None), path

print("FINITE-STATE MORPHOLOGICAL PARSER - VERB INFLECTION\n")
for w in words:
    final, path = parse(w)
    print("Input word      :", w)
    print("State path      :", " -> ".join(path))
    if final:
        tag, kind, morphs, desc = ACCEPT[final]
        print("Morphemes       :", morphs)
        print("Inflection      :", tag, "-", desc)
        print("Pattern         :", kind)
        print("Root            :", ROOT)
        print("Normalized form :", ROOT + "+" + tag)
    else:
        print("Result          : REJECTED (not a valid form)")
    print("-" * 60)

# Summary table
print("\n{:<9}{:<12}{:<10}{:<8}{}".format("Word", "Pattern", "Root", "Tag", "Final State"))
for w in words:
    final, _ = parse(w)
    print("{:<9}{:<12}{:<10}{:<8}{}".format(w, ACCEPT[final][1], ROOT, ACCEPT[final][0], final))

FINITE-STATE MORPHOLOGICAL PARSER - VERB INFLECTION

Input word      : writes
State path      : q0 -> q1 -> q2 -> q3 -> q4 -> q5 -> q6
Morphemes       : write + -s
Inflection      : VBZ - 3rd person singular present
Pattern         : Regular
Root            : write
Normalized form : write+VBZ
------------------------------------------------------------
Input word      : writing
State path      : q0 -> q1 -> q2 -> q3 -> q4 -> q7 -> q8 -> q9
Morphemes       : write + -ing
Inflection      : VBG - present participle (final 'e' dropped before -ing)
Pattern         : Regular
Root            : write
Normalized form : write+VBG
------------------------------------------------------------
Input word      : written
State path      : q0 -> q1 -> q2 -> q3 -> q4 -> q10 -> q11 -> q12
Morphemes       : write + -en
Inflection      : VBN - past participle (stem change: writ + t + en)
Pattern         : Irregular
Root            : write
Normalized form : write+VBN
----------------------------------------

In [6]:
# Q5: Porter Stemmer (step by step) for document retrieval
# Words: relational, relation, relate   (runs as-is in Google Colab, no libraries needed)

words = ["relational", "relation", "relate"]

def is_cons(w, i):
    if w[i] in "aeiou":
        return False
    if w[i] == "y":
        return i == 0 or not is_cons(w, i - 1)
    return True

def measure(stem):                       # m in [C](VC){m}[V]
    form = "".join("C" if is_cons(stem, i) else "V" for i in range(len(stem)))
    return form.replace("VC", "|").count("|") if form else 0

def has_vowel(stem):
    return any(not is_cons(stem, i) for i in range(len(stem)))

def double_cons(w):
    return len(w) > 1 and w[-1] == w[-2] and is_cons(w, len(w) - 1)

def cvc(w):                              # consonant-vowel-consonant, last not w, x, y
    return (len(w) >= 3 and is_cons(w, len(w) - 1) and not is_cons(w, len(w) - 2)
            and is_cons(w, len(w) - 3) and w[-1] not in "wxy")

def replace(w, rules, min_m):
    """Apply the longest matching suffix rule if m(stem) > min_m."""
    for suf, rep in sorted(rules, key=lambda r: -len(r[0])):
        if w.endswith(suf):
            stem = w[:-len(suf)]
            if measure(stem) > min_m:
                return stem + rep, f"{suf} -> '{rep}'"
            return w, None
    return w, None

def step1a(w):
    for suf, rep in [("sses", "ss"), ("ies", "i"), ("ss", "ss"), ("s", "")]:
        if w.endswith(suf):
            return w[:-len(suf)] + rep, f"{suf} -> '{rep}'"
    return w, None

def step1b(w):
    if w.endswith("eed"):
        return (w[:-1], "eed -> ee") if measure(w[:-3]) > 0 else (w, None)
    for suf in ("ed", "ing"):
        if w.endswith(suf) and has_vowel(w[:-len(suf)]):
            w = w[:-len(suf)]
            if w.endswith(("at", "bl", "iz")): w += "e"
            elif double_cons(w) and w[-1] not in "lsz": w = w[:-1]
            elif measure(w) == 1 and cvc(w): w += "e"
            return w, f"{suf} removed"
    return w, None

def step1c(w):
    if w.endswith("y") and has_vowel(w[:-1]):
        return w[:-1] + "i", "y -> i"
    return w, None

STEP2 = [("ational","ate"),("tional","tion"),("enci","ence"),("anci","ance"),("izer","ize"),
         ("abli","able"),("alli","al"),("entli","ent"),("eli","e"),("ousli","ous"),
         ("ization","ize"),("ation","ate"),("ator","ate"),("alism","al"),("iveness","ive"),
         ("fulness","ful"),("ousness","ous"),("aliti","al"),("iviti","ive"),("biliti","ble")]
STEP3 = [("icate","ic"),("ative",""),("alize","al"),("iciti","ic"),("ical","ic"),("ful",""),("ness","")]
STEP4 = [(s, "") for s in ("al","ance","ence","er","ic","able","ible","ant","ement","ment",
                           "ent","ion","ou","ism","ate","iti","ous","ive","ize")]

def step2(w): return replace(w, STEP2, 0)
def step3(w): return replace(w, STEP3, 0)

def step4(w):
    for suf, _ in sorted(STEP4, key=lambda r: -len(r[0])):
        if w.endswith(suf):
            stem = w[:-len(suf)]
            if suf == "ion" and not stem.endswith(("s", "t")):
                return w, None
            return (stem, f"{suf} removed") if measure(stem) > 1 else (w, None)
    return w, None

def step5a(w):
    if w.endswith("e"):
        stem = w[:-1]
        if measure(stem) > 1 or (measure(stem) == 1 and not cvc(stem)):
            return stem, "e removed"
    return w, None

def step5b(w):
    if measure(w) > 1 and double_cons(w) and w.endswith("l"):
        return w[:-1], "ll -> l"
    return w, None

STEPS = [("Step 1a", step1a), ("Step 1b", step1b), ("Step 1c", step1c), ("Step 2", step2),
         ("Step 3", step3), ("Step 4", step4), ("Step 5a", step5a), ("Step 5b", step5b)]

def porter(word):
    log, w = [], word.lower()
    for name, fn in STEPS:
        new, rule = fn(w)
        if rule:
            log.append((name, rule, w, new))
        w = new
    return w, log

print("PORTER STEMMER - DOCUMENT RETRIEVAL\n")
summary = []
for word in words:
    final, log = porter(word)
    print(f"Word: {word}   (m of word = {measure(word)})")
    for name, rule, before, after in log:
        print(f"   {name:<8}| rule: {rule:<18}| {before} -> {after}")
    if not log:
        print("   no rule applied")
    print(f"   FINAL STEM: {final}\n")
    summary.append((word, " ; ".join(f"{n}: {r}" for n, r, _, _ in log), final))

print("{:<12}{:<50}{}".format("Original", "Rules Applied", "Final Stem"))
print("-" * 72)
for word, rules, final in summary:
    print("{:<12}{:<50}{}".format(word, rules, final))
print("\nAll variants share the stem:", {s[2] for s in summary})

PORTER STEMMER - DOCUMENT RETRIEVAL

Word: relational   (m of word = 4)
   Step 2  | rule: ational -> 'ate'  | relational -> relate
   Step 5a | rule: e removed         | relate -> relat
   FINAL STEM: relat

Word: relation   (m of word = 3)
   Step 2  | rule: ation -> 'ate'    | relation -> relate
   Step 5a | rule: e removed         | relate -> relat
   FINAL STEM: relat

Word: relate   (m of word = 2)
   Step 5a | rule: e removed         | relate -> relat
   FINAL STEM: relat

Original    Rules Applied                                     Final Stem
------------------------------------------------------------------------
relational  Step 2: ational -> 'ate' ; Step 5a: e removed     relat
relation    Step 2: ation -> 'ate' ; Step 5a: e removed       relat
relate      Step 5a: e removed                                relat

All variants share the stem: {'relat'}
